# Room Graph
Builds a room-to-room graph from wall surfaces.
- **1 node = 1 room**
- **1 edge = shared wall (adjacency) or door/window (access)**

In [32]:
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper

print(Helper.Version())
renderer = "vscode"

The version that you are using (0.9.18) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 1. Load geometry

In [33]:
geo_objects = Topology.ByOBJPath("c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Geometry _etm.obj")
print(f"{len(geo_objects)} object(s) loaded")

Topology.Show(geo_objects,
              faceColor=[210,210,250],
              faceOpacity=0.3,
              edgeColor="white",
              edgeWidth=1,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

1 object(s) loaded


## 2. Extract faces and detect rooms

In [34]:
all_faces = []
for obj in geo_objects:
    all_faces.extend(Topology.Faces(obj))
print(f"{len(all_faces)} faces extracted")

cellcomplex = None
for tol in [0.01, 0.1, 1.0]:
    cellcomplex = CellComplex.ByFaces(all_faces, tolerance=tol)
    if cellcomplex is not None:
        rooms = Topology.Cells(cellcomplex)
        print(f"tolerance={tol} → {len(rooms)} room(s) detected")
        break

if cellcomplex is None:
    print("ERROR: No enclosed volumes found. Walls may have gaps in Rhino.")

317 faces extracted
tolerance=0.01 → 10 room(s) detected


## 3. Show rooms (each colour = one room)

In [35]:
rooms = Topology.Cells(cellcomplex)
palette = Color.ColorRamp(len(rooms))

for i, room in enumerate(rooms):
    room = Topology.SetDictionary(room, Dictionary.ByKeysValues(["color"], [palette[i]]))

Topology.Show(cellcomplex,
              faceColorKey="color",
              faceOpacity=0.5,
              edgeColor="white",
              edgeWidth=1,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

AttributeError: type object 'Color' has no attribute 'ColorRamp'

## 4. Adjacency graph — connected through shared walls

In [ ]:
g_adj = Graph.ByTopology(cellcomplex)
print(f"Adjacency: {len(Graph.Vertices(g_adj))} nodes, {len(Graph.Edges(g_adj))} edges")

for v in Graph.Vertices(g_adj):
    v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_adj):
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "white"]))

Topology.Show(cellcomplex, g_adj,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 5. Load apertures (doors & windows)

In [ ]:
apt_objects = Topology.ByOBJPath("c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Apertures _etm.obj")
apertures = []
for obj in apt_objects:
    apertures.extend(Topology.Faces(obj))
print(f"{len(apertures)} aperture face(s) loaded")

cc_apts = Topology.AddApertures(cellcomplex, apertures, subTopologyType="face")

## 6. Access graph — connected only through doors/windows

In [ ]:
g_access = Graph.ByTopology(cc_apts, direct=False, viaSharedApertures=True)
print(f"Access: {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")

for v in Graph.Vertices(g_access):
    v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g_access):
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "#2CFF96"]))

Topology.Show(cc_apts, g_access,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)